# Phytoplankton Phenology Study in the Northwest Atlantic Ocean

#### Import Python Libraries and datasets

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.patheffects as path_effects
import cartopy
import pandas as pd
from shapely.ops import unary_union
import cartopy.feature as cfeature
import geopandas as gpd
from shapely.geometry import mapping
from shapely.geometry import Polygon
import rioxarray
#import sys
#sys.path.append(r'C:\Users\grace.davis\Documents\GitHub\RESOURCES\python')
#import utilities
#from utilities import get_prod_files
import cmocean
from matplotlib.colors import LogNorm
import cartopy.crs as crs
import statsmodels as sm
from statsmodels import nonparametric
from statsmodels.nonparametric import smoothers_lowess
import scipy
from scipy import signal
from scipy.signal import find_peaks
from scipy.signal import argrelextrema
from scipy.signal import savgol_filter
from scipy.integrate import trapezoid
from collections import Counter
import plotly.express as px
import seaborn as sns
import calendar
from scipy import stats
from collections import defaultdict

In [2]:
#Daily chlorophyll data and regional zarr files
daily_data = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr',consolidated=True)
MABS = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\MABS_D8.zarr')
MABN = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\MABN_D8.zarr')
GB = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\GB_D8.zarr')
GOMW = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\GOMW_D8.zarr')
GOME = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\GOME_D8.zarr')

In [3]:
#Region shapefiles
shapefile = gpd.read_file('https://github.com/hsynan/READ-EDAB-Synan_hydrographic_climatologies/raw/refs/heads/main/data/shapefiles/NES_5REGIONS.zip')
MAB_south_loc = shapefile.iloc[[0]].set_crs("epsg:4326", inplace=True)
MAB_north_loc = shapefile.iloc[[1]].set_crs("epsg:4326", inplace=True)
GB_whole_loc = shapefile.iloc[[2]].set_crs("epsg:4326", inplace=True)
GOM_west_loc = shapefile.iloc[[3]].set_crs("epsg:4326", inplace=True)
GOM_east_loc = shapefile.iloc[[4]].set_crs("epsg:4326", inplace=True)
NES = shapefile.dissolve()

### Rate of Change Method
The rate of change method tracks the daily rate of change in chlorophyll-a concentration and finds the maximum rates of change throughout the time series. These maximum rates of change identify phytoplankton blooms. Initiation and termination dates are determined by inflection points in the chlorophyll rate of change. This method does not rely on climatological values, it focuses on a daily differences and compares them to other values after a set period of days. You can set the number of days between blooms in order to eliminate extra peaks from small spikes in the bloom. Prior to the rate of change being calculated, the chlorophyll-a data should be smoothed so rates of change accurately represent blooms.

#### Regional Rates of Change

In [ ]:
def bounding_data(dataset,shapefile_geometry):
    """
    Regionally subsets a dataset for general analysis.

    This function takes a shapefile geometry and subsets a larger dataset to only include data within the shapefile. The data is averaged along the lat and lon dimensions.

    Args:
        dataset (xarray.Dataset, required): General dataset in question. No defaults
        shapefile_geometry (geopandas.GeoDataFrame, required): Shapefile of the region in question. No defaults

    Returns:
        xarray.DataArray. The arrays of spatially sliced data.
    """
    dataset.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
    dataset.rio.write_crs("epsg:4326", inplace=True)
    clipped_daily = dataset.rio.clip(shapefile_geometry.geometry.apply(mapping), shapefile_geometry.crs, drop=True)
    regional_year = clipped_daily.CHL_median.mean(dim=['lat','lon'])
    return regional_year

In [ ]:
def smoothing_data(dataset=None, path=None, var_name='CHL_median', dim='time', method="SavGol", window=15, poly=3, deriv=0, frac=0.00117):
    """
    Smoothes the raw chlorphyll-a data using a specific smoothing technique.

    If no method is provided, the default is the Savistky-Golay technique which has default parameters of a 15 day window and a polyorder of 3.
    If method is provided as "lowess", the frac value defaults to 0.00117, equivalent of a 12 day window on a 27 year time series.
    If no dataset path is provided, the function searches for D8 CHL files for the NES region. Currently, it opens the zarr file, but can be uncomment to open netCDFs.

    Args:
        dataset (xarray.Dataset, optional): A spatially averaged dataset. Default is None
        path (str, optional): Path to a dataset. Defaults to daily D8 data for the full time series.
        var_name (str, optional): Name of variable of interest in the dataset. Defaults to 'CHL_median'
        dim (str, optional): The dimension to interpolate and smooth across. Defaults to 'time'
        method (str, optional): Smoothing technique applied. Defaults to "SavGol" but can also receive "lowess".
        window (int, optional): Window for SavGol smoothing. Default is 15
        poly (int, optional): polyorder for SavGol smoothing. Default is 3
        deriv (int, optional): Derivative of SavGol function. 0 provides smoothed data and 1 provides the first derivative. Defaults to 0 
        frac (float, optional): Frac value for lowess smoothing. Only necessary for using lowess smoothing. Default is 0.00117

    Returns:
        numpy.ndarray. Array of smoothed chlorophyll data values.
    """
    if dataset is not None:
        data = dataset
    elif path is None:
        data = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_combined.zarr')
    else:
        data = xr.open_mfdataset(path)
    
    if isinstance(data, xr.Dataset):
        chl = data[var_name]
    else:
        chl = data

    #Savitsky-Golay smoothing
        #Requires no NaN values so it uses a linear interpolation to fill values.
    if method == "SavGol":
        data_filled = (chl.interpolate_na(dim=dim, method='linear', fill_value='extrapolate'))
        #Does the smoothing for each point in the spatial data along the time dimension. Should work as long as there is a time dimension.
        smoothed_CHL = xr.apply_ufunc(
            scipy.signal.savgol_filter,
            data_filled,
            kwargs={
                "window_length": window,
                "polyorder": poly,
                "deriv": deriv,
                "axis": -1,
            },
            input_core_dims=[[dim]],
            output_core_dims=[[dim]],
            dask='parallelized',
            output_dtypes=[chl.dtype]
        )
        smoothed_median = smoothed_CHL.where(~np.isnan(chl))
    #LOWESS smoothing
        #LOWESS function handles NaN values so no interpolation is needed.
    elif method == "lowess":
        def apply_lowess(y):
            mask = ~np.isnan(y)
            if mask.sum() < 3:
                return np.full_like(y, np.nan)
            smoothed = sm.nonparametric.smoothers_lowess.lowess(
                endog=y[mask],
                exog=np.arange(len(y))[mask],
                frac=frac,
                return_sorted=False
            )
            out = np.full_like(y, np.nan, dtype=np.float64)
            out[mask] = smoothed
            return out
        smoothed_median = xr.apply_ufunc(
            apply_lowess,
            chl,
            input_core_dims=[[dim]],
            output_core_dims=[[dim]],
            vectorize=True,
            dask="parallelized",
            output_dtypes=[chl.dtype]
        )
    else:
        raise ValueError("Error: Must specify smoothing technique")
    return smoothed_median

First, we identify chlorophyll maximums in the dataset. To start, we also make sure that the chlorophyll peaks are significant, so the maximum chlorophyll concentrations must exceed the 10% threshold designated from testing the threshold method. 

In [ ]:
def bloom_peak_detection_1D(clipped_thld,dataset,window_for_peak=10,days=14,prm=0.1):
    """
    Detects all peak chlorophyll values that exceed the climatological threshold for a 1-D array

    This function uses the smoothed chlorophyll data, identified peak values, and then masks that data to include only peaks that exceed the threshold set by the climatology.
    This function uses the find_peaks function from scipy, as well as the threshold_value() function and smoothing_data() function. 
    The clipped_thld variable is created in the rolling_peak_window function or as a global variable.

    Args:
        clipped_thld (float, required): Pre-calculated climatological threshold value for the region of interest. No defaults
        dataset (xarray.Dataset, required): Already smoothed dataset. Defaults to None
        window_for_peak (int, optional): The number of days the chlorophyll must remain above the threshold to be considered a peak. Default is 10
        days (int, optional): Distance variable for scipy find_peaks. Distance allowed between consecutive peaks. Default is 14
        prm (float, optional): Prominence variable for find_peaks. Percent above the other peaks to be considered a peak. Default is 0.1
    
    Returns:
        List. List of days since the start of the dataset where the chlorophyll peaked and was above the threshold.
    """
    # STEP 1: Define the dataset. Uses a smoothed dataset (if provided). Else, it smooths the raw data provided for the region of interest.
    smoothed_CHL = np.asarray(dataset).ravel()
    if np.isnan(smoothed_CHL).all():
        return np.zeros(smoothed_CHL.shape, dtype=bool)
    thld = float(np.asarray(clipped_thld).flat[0])
    # STEP 2: Find chlorophyll peaks with find peaks function.
        # Default of 14 days for distance was chosen after testing distances from 7-31
        # Default prominence of 0.1 captures major blooms while ignoring small peaks from daily fluctuations/sensor noise. Tested values in range of 0.01 - 0.2. 
    chl_peak_loc, _ = find_peaks(smoothed_CHL,distance=days,prominence=prm) #Finds all peaks
    chl_peaks = []
    chl_series = pd.Series(smoothed_CHL) #Turns chlorophyll values into a pandas series

    # STEP 3: Create a Boolean list for values that surpass/do not exceed the set threshold.
    is_above_threshold = chl_series>thld #Creates a true and false list. True if the value exceeds the threshold.

    # STEP 4: Searches Boolean list for places where the value switches from True to False (or False to True)
    change_from_prev_day = is_above_threshold != is_above_threshold.shift() #Checks if there is a change from previous day

    # STEP 5: Create streak IDs for each event and group events with the same ID together
    streak_IDs = change_from_prev_day.cumsum() #Creates ID for each event (New ID starts when the Boolean value changes. If no change, the ID is the same for that day)
    streak_lengths = is_above_threshold.groupby(streak_IDs).transform('sum') #Groups events together with the same ID and calculates the number of days that share that ID
    
    # STEP 6: Check to ensure the peak is above the threshold and check to see if its streak ID is >= to the defined window_for_peak.
        # If both conditions are true, we add it to the chl_peaks list. If one or both is not met, the peak is discarded.
    for peak in chl_peak_loc:
        peak_above_threshold = is_above_threshold[peak] #Checks that the peak is above the threshold
        peak_length = streak_lengths[peak]>=window_for_peak #Checks that the chlorophyll values remain above the threshold for a specified window
        if peak_above_threshold and peak_length:
            chl_peaks.append(peak)
    mask = np.zeros(len(dataset), dtype=bool)
    mask[chl_peaks] = True
    return mask

In [ ]:
def bloom_peak_detection(data, clipped_thld, var_name='smoothed_CHL', time_dim='time', window_for_peak=10, days=14, prm=0.1, return_indices_for_1d=True):
    """
    Identifies the chlorophyll peaks in the dataset.

    Args:
        data (xr.Dataset, required): The dataset for the chlorophyll data. No defaults
        clipped_thld (float, required): The threshold value for the pixels/region. No defaults
        var_name (str, optional): The name of the smoothed chlorophyll variable in the dataset. Defaults to 'smoothed_CHL'
        time_dim (str, optional): The time dimension name in the dataset. Defaults to 'time'
        window_for_peak (int, optional): The number of days the chlorophyll must be exceeding the threshold to be considered a peak. Defaults to 10
        days (int, optional): The minimum number of days between peaks. Defaults to 14
        prm (float, optional): The minimum prominence of the peaks compared to surrounding chlorophyll. Defaults to 0.1
        return_indices_for_1d (Bool, optional): Determines whether the function also returns a list of integers or a xr.Dataset with dimensions and coordinates. Defaults to True

    Returns:
        List, xr.DataArray, pd.Series, or np.ndarray: Output type depends on 'return_indices_for_1d' and type of 'chl':
            - List: List of integer indices where chlorophyll peaked above the threshold. Returned if 'return_indices_for_1d' is True
            - xr.DataArray: Boolean mask matching the original coordinates and dimensions of `chl`. Returned if `return_indices_for_1d` is False and `chl` is an xarray DataArray.
            - pd.Series: Boolean mask matching the original index of `chl`. Returned if `return_indices_for_1d` is False and `chl` is a pandas Series.
            - np.ndarray: Raw 1-D boolean array mask. Returned if `return_indices_for_1d` is False and `chl` is any other array type.
    """
    # STEP 1: Opens datasets. Can take a xarray Dataset, DataArray, pandas DataFrame, Series, or numpy array. 
    if isinstance(data, (xr.Dataset, pd.DataFrame)):
        #If the data is a dataset or dataframe, the user must specify the variable name.
        if var_name in data:
            chl = data[var_name]
        else:
            raise ValueError("Must specify variable name")
    elif isinstance(data, (xr.DataArray, pd.Series, np.ndarray)):
        chl = data
    else:
        raise TypeError("Unsupported data type")
    # STEP 2a: For multi-dimensional datasets (specifically with spatial data),we apply bloom_peak_detection_1D function along the time dimension for each pixel.
    if getattr(chl, 'ndim', 1) > 1:
        peak_mask = xr.apply_ufunc(
            bloom_peak_detection_1D,
            chl,
            clipped_thld,
            input_core_dims=[[time_dim], []],
            output_core_dims=[[time_dim]],
            kwargs={
                'window_for_peak': window_for_peak,
                'days': days,
                'prm': prm
            },
            vectorize=True,
            dask='parallelized',
            output_dtypes=[bool],
            dask_gufunc_kwargs={'allow_rechunk':True} #Allows for rechunking of the data to optimize parallel processing
        )
        return peak_mask.transpose(*chl.dims)
    # STEP 2b: If the data is one dimensional (no spatial dimensions), it applies the bloom_peak_detection_1D function, returning either a list of indices or a boolean mask.
    else:
        if isinstance(chl, (xr.DataArray, pd.Series)):
            values = chl.values
        else:
            values = np.asarray(chl)
        thld_values = float(clipped_thld.values) if isinstance(clipped_thld, xr.DataArray) else clipped_thld
        mask_1d = bloom_peak_detection_1D(
            dataset=values,
            clipped_thld=thld_values,
            window_for_peak=window_for_peak,
            days=days,
            prm=prm
        )
        if return_indices_for_1d: #Returns a list of boolean values if False.
            return np.where(mask_1d)[0].tolist()
        else:
            if isinstance(chl, xr.DataArray):
                return xr.DataArray(mask_1d, coords=chl.coords, dims=chl.dims)
            elif isinstance(chl, pd.Series):
                return pd.Series(mask_1d, index=chl.index)
            return mask_1d

In [ ]:
def bloom_event_detection(clipped_thld,chl_peaks_list,dataset,var_name='smoothed_CHL',event_distance=21,peak_window=10):
    """
    This function finds peaks in the chlorophyll-a time series and then groups together peaks in the same event based on proximity.

    If there are no peaks, the function returns an empty list.
    Using a list of chlorophyll peaks from the bloom_peak_detection function (previously calculated), the function searches for peaks in close proximity.
    A ten day rolling window is created for each peak to see if the chlorophyll value drop below the climatological threshold. If it does, the loop breaks.
    If peaks are too close together or the chlorophyll value does not drop below the threshold, they are considered one event. If these conditions are not met, they are separate events.

    Args:
        clipped_thld (variable, required): Threshold value for the region of interest. No defaults
        chl_peaks_list (list, required): Pre-calculated list of chlorophyll peaks for dataset. No defaults
        dataset (variable, required): Already smoothed dataset. No defaults
        var_name (str, optional): The variable name for the smoothed chlorophyll dataset. Defaults to 'smoothed_CHL'
        event_distance (int, optional): The number of days peaks must be apart to be considered separate events. Defaults to 21
        peak_window (int, optional): The number of days the chlorophyll concentration must remain above or below the threshold. Defaults to 10

    Returns: 
        List: List of bloom event and peaks within each event.
    """
    # STEP 1: Load in chlorohyll peak list
    chl_peaks = [int(p) for p in chl_peaks_list]
    if len(chl_peaks) == 0: #Returns empty list if no peaks were found
        return []
    thld_val = (float(clipped_thld.values) if isinstance(clipped_thld, xr.DataArray) else float(clipped_thld))

    # STEP 2: Defines the smoothed dataset
    if isinstance(dataset, (xr.Dataset, pd.DataFrame)):
        if var_name in dataset:
            smoothed_CHL = np.asarray(dataset[var_name]).squeeze()
        else:
            raise ValueError("Must specify correct variable name")
    elif isinstance(dataset, (xr.DataArray, pd.Series)):
        smoothed_CHL = dataset.values.squeeze()
    else:
        smoothed_CHL = np.squeeze(np.asarray(dataset))
    bloom_events = []

    # STEP 3: Creates the range for chlorophyll values to be observed in and identifies peak timeline
    current_event = [chl_peaks[0]] #Current event starts at the first peak identified
    days_between_events = event_distance #The number of days that must pass between conditions for the peaks to be considered separate events
    for i in range(1,len(chl_peaks)):
        previous_peak = int(chl_peaks[i-1]) #Finds the previous peak
        current_peak = int(chl_peaks[i])
        chl_between_peaks = smoothed_CHL[previous_peak:current_peak] #Creates a list of all chlorophyll values between the current peak and previous peak
        dropped_below_thld = False

    # STEP 4: Find if the chlorophyll concentration drops below the threshold for a certain number of consecutive days
        #Checks to see if the number of days between chlorophyll peaks is above the specified peak window
        if len(chl_between_peaks)>=peak_window:
            chl_series = pd.Series(chl_between_peaks)
            #If all chlorophyll values are below the pre-determined threshold, dropped_below_thld is true. It adds up the trues and falses and finds the spots where the value is equal to peak_window
            dropped_below_thld = (chl_series<thld_val).rolling(window=peak_window).sum().eq(peak_window).any()
    # STEP 5: Append events to events list. 
        if current_peak-previous_peak<days_between_events or not dropped_below_thld: #If peaks are too close together or does not drops below threshold, they are the same event.
            current_event.append(current_peak)
        else: #Peaks are an appropriate distance apart or chl drop below the threshold.
            bloom_events.append(current_event)
            current_event = [current_peak]
    bloom_events.append(current_event)
    return bloom_events


Next, from the event detection above, we find the largest peak in each event and the maximum rate of change prior to the largest peak.

In [ ]:
def max_peak(event, dataset, time_array, var_name_smooth='smoothed_CHL'):
    """
    Finds the peak in an event with the maximum chlorophyll concentration for the event.
    For use in bloom_timing function.

    Args:
        event (list, required): Pre-calculated list of peaks for the event. No defaults
        dataset (xarray.Dataset, required): The dataset for analysis. No defaults
        var_name_smooth (str, optional): The variable name for the smoothed chlorophyll data. Defaults to 'smoothed_CHL'.

    Returns:
        List. A list of peaks associated with that blooms maximum chlorophyll concentration.
    """
    if hasattr(dataset, var_name_smooth):
        region_smoothed = dataset[var_name_smooth].values
    elif hasattr(dataset, 'values'):
        region_smoothed = dataset.values
    else:
        region_smoothed = np.asarray(dataset)
    region_time_smoothed = time_array #Extracts time values
    peak_chl_values = -float('inf')
    max_chl_day = None
    flatten_event = [] #For events that are multimodal, this flattens it into one list and not a tuple
    for item in event:
        if isinstance(item,(tuple,list,np.ndarray)): #If the event has more than one peak, it makes it one list and not a variety of data types.
            flatten_event.extend(item)
        else:
            flatten_event.append(item)
    for day in flatten_event:
        chl_peak = region_smoothed[int(day)] #Finds the chl value at the peak
        if chl_peak>peak_chl_values: #Checks to see if the current chl value is greater than the previous peak's value.
            peak_chl_values = chl_peak #If it is, it becomes the new maximum of the event
            max_chl_day = day #This is the day of the maximum
    peak_DOY = region_time_smoothed[max_chl_day]
    if isinstance(region_smoothed, np.ndarray):
        peak_chl = region_smoothed[max_chl_day]
    else:
        peak_chl = region_smoothed.values[max_chl_day]
    ts = pd.Timestamp(peak_DOY)
    peak_date = ts.date()
    peak_doy = ts.dayofyear
    return peak_date,peak_doy,peak_chl

In [ ]:
def max_roc_for_bloom(start_DOY,end_DOY,dataset,roc_var_name='ROC'):
    """
    Finds the maximum rate of change for each bloom.

    This function uses a pre-saved dataset of daily rates of change and pre-calculated start and end DOYs for each event
    It then finds the maximum rate of change between the initiation and termination date and then adds it to the start day value to get the DOY value for the maximum rate of change.
    This function is part of the bloom_timing function.

    Args:
        start_DOY (int, required): Pre-calculated start DOY for the event. No defaults
        end_DOY (int, required): Pre-calculated end DOY for the event. No defaults
        dataset (xarray.Dataset, required): Dataset of interest. No defaults
        roc_var_name (str, optional): Variable name for the rate of change of daily chlorophyll. Defaults to 'ROC'
    Returns:
        int/float: The maximum rates of change for the bloom.
    """
    roc = dataset[roc_var_name]
    range_roc = roc[start_DOY:end_DOY+1]
    if len(range_roc)>0:
        range_max_roc = np.nanargmax(range_roc) #Finds local maximum rate of change for each detected bloom
        max_roc = start_DOY+range_max_roc #Gets the actual day of year value
    max_roc = max_roc
    return max_roc

This large function finds a variety of bloom timing characteristics based on the rate of change.

In [ ]:
def bloom_timing(clipped_thld,clipped_med,dataset,data_type='daily',var_name_smooth='smoothed_CHL',roc_var_name='ROC',init_term_window=5,trough_length=3,search_window=90,**kwargs):
    """
    This function finds the initiation and termination dates of blooms based on a rolling peak window.

    This function finds a variety of bloom timing values, including:
        - Bloom events: The events (and peaks within the events) throughout the dataset that satisfy the peak conditions.
        - Start day: The day since the start of the dataset where the initiation conditions were met for each bloom event.
        - End day: The day since the start of the dataset where the termination conditions were met for each bloom event
        - Peak date: The date of the maximum chlorophyll peak for each bloom event.
        - Peak DOY: The DOY (1-366) of the maximum chlorophyll peak for each bloom event.
        - Maximum chlorophyll: The maximum chlorophyll value for each bloom event.
        - Maximum rate of change: The maximum rate of change for each bloom event.
        - First exceedance day: The day for each bloom event where the chlorophyll concentration first exceed the pre-determined threshold.
        - Last dip day: The day for each bloom event where the chlorophyll concentration last dipped below the pre-determined threshold.


    Args:
        clipped_thld (float, required): Pre-calculated climatological threshold for the region. No defaults
        clipped_med (float, required): Pre-calculated climatological median for the region. No defaults
        dataset (xarray.Dataset, required): The daily chlorophyll dataset. No defaults
        data_type (str, optional): The type of data being used. Accepts 'daily' or 'climatology'. Defautls to 'daily'
        var_name_smooth (str, optional): The variable name for the smoothed chlorophyll data in the dataset. Defaults to 'smoothed_CHL'
        roc_var_name (str, optional): The variable name for the daily rate of change. Defaults to 'ROC'
        init_term_window (int, optional): Amount of time each condition must be met for it to trigger an initiation or termination date. Defaults to 5
        trough_length (int, optional): The number of days on either side of a minimum chl value that the value must remain below for it to be considered a trough. Defaults to 3
        search_window (int, optional): The number of days after the final peak of an event that the function searches through to find the termination date. Defaults to 90
        **kwargs: Additional input for bloom_event_detection and threshold_value functions. Possible inputs include: 
            - peak_window (int, optional): The amount of time a peak must remain above the threshold for it to be considered an event. Defaults to 10  
            - days (int, optional): Distance for find_peaks function. Defaults to 10
            - prm (float, optional): Prominence for find_peaks function. Defaults to 0.01

    Returns:
        tuple: A tuple containing (start_DOY, end_DOY, merge_bloom_events, peak_dates, peak_DOYs, maximum_chl, maximum_roc,start_at_thld_list,end_at_thld_list), where:
            start_DOY (list): The list of bloom initiation days.
            end_DOY (list): The list of bloom termination days.
            merge_bloom_events (list): The list of bloom events.
            peak_dates (list): The list of chlorophyll maximum days for each bloom event.
            peak_DOYs (list): The list of dates of the chlorophyll maximums for each bloom event.
            maximum_chl (list): The list of maximum chlorophyll values for each bloom event.
            maximum_roc (list): The list of maximum rates of change for each bloom event.
            start_at_thld_list (list): The list of DOYs where the chl first crosses the threshold for an event.
            end_at_thld_list (list): The list of DOYs where the chl last dipped below the threshold for an event.
    """
    # STEP 1: Identify chlorophyll median dataset and find the rate of change, bloom peaks, and bloom events.
    if isinstance(dataset, (xr.Dataset, pd.DataFrame)):
        if var_name_smooth in dataset:
            chl_median = np.asarray(dataset[var_name_smooth]).squeeze()
        else:
            raise ValueError("Must specify correct variable name")
    elif isinstance(dataset, (xr.DataArray, pd.Series)):
        chl_median = np.asarray(dataset).squeeze()
    else:
        chl_median = np.asarray(dataset).squeeze()

    if data_type == 'daily' and isinstance(dataset, (xr.Dataset, pd.DataFrame)):
        if roc_var_name in dataset:
            roc = np.asarray(dataset[roc_var_name]).squeeze()
        else:
            roc = np.gradient(chl_median)
    elif data_type == 'climatology' and isinstance(dataset, (xr.Dataset, pd.DataFrame)):
        if roc_var_name in dataset:
            roc = np.asarray(dataset[roc_var_name]).squeeze()
        else:
            roc = np.gradient(chl_median)
    chl_median_series = pd.Series(chl_median)
    chl_peaks_list = bloom_peak_detection(clipped_thld=clipped_thld,data=dataset,var_name=var_name_smooth,**kwargs)
    bloom_events = bloom_event_detection(dataset=dataset,clipped_thld=clipped_thld,chl_peaks_list=chl_peaks_list,var_name=var_name_smooth,**kwargs)
    
    # STEP 2: Define peak windows
    last_end_day, last_start_day = 0, 0
    start_DOY, end_DOY, merge_bloom_events = [], [], []
    peak_dates, peak_DOYs, maximum_chl = [], [], []
    maximum_roc = []
    max_index = len(roc)-1

    # STEP 3: Identify all possible initiation and termination dates for the full time series
    is_roc_negative =  roc < 0
    is_roc_positive = roc >= 0
    is_below_threshold = chl_median < clipped_thld
    is_below_median = chl_median <= clipped_med

    #Find all days for time series where initiation conditions are met (positive growth and below the threshold)
    initiation_conditions_met = pd.Series(is_roc_positive & is_below_threshold)
    initiation_rolling = initiation_conditions_met.rolling(window=init_term_window).sum()
    initiation_days = initiation_rolling[initiation_rolling == init_term_window].index.to_numpy()

    #Find all days for time series where termination conditions are met
    termination_conditions_met = pd.Series(is_roc_negative & is_below_threshold)
    termination_rolling = termination_conditions_met.rolling(window=init_term_window).sum()
    termination_days = termination_rolling[termination_rolling == init_term_window].index.to_numpy()

    #Find all local troughs for the full dataset. The troughs must have chl values lower than specified consecutive days on either side. 
    #This smooths out some of the smaller bumps caused by the noisy chl-a data and keeps major troughs. Tested 1,2,3. 
    chl_median_series = pd.Series(chl_median)
    local_minimum = (chl_median_series < chl_median_series.shift(trough_length)) & (chl_median_series < chl_median_series.shift(-trough_length)) & is_below_median
    local_minimum = local_minimum[local_minimum].index.to_numpy()

    # STEP 4: Find the initiation date of the bloom based on the rate of change.
    for event in bloom_events:
        event_start = event[0]
        event_end = event[-1]

        #Sets the end of the window to be 90 days from the last peak in the event or the end of the dataset, whichever comes first.
        end_of_window = min(max_index,event[-1]+search_window) 
        start_day = last_end_day #Sets the start of the window to the last end day

        #Find all possible initiation days between the end of the last bloom and the first peak in the current event.
        possible_init_dates = initiation_days[(initiation_days < event_start)&(initiation_days >= start_day)] 

        if len(possible_init_dates) > 0 and last_end_day >= last_start_day: #Ensures that the initiation date is not before the previous bloom's termination date
            possible_start_day = possible_init_dates[-1] - init_term_window + 1
            window_troughs = local_minimum[(local_minimum >= last_end_day) & (local_minimum <= event_start)] #Finds all troughs between first peak and the previous termination

            #Attach it to the closest trough if one is available
            if len(window_troughs) > 0:
                 closest_trough = np.abs(window_troughs - possible_start_day).argmin() #Finds the closest trough to the first peak
                 start_day = window_troughs[closest_trough] 
            else:
                 start_day = possible_start_day  

    # STEP 5: Find the termination date 
        end_day = event_end
        possible_term_dates = termination_days[(termination_days>event_end) & (termination_days<=end_of_window)]

        if len(possible_term_dates)>0:
            possible_end_day = possible_term_dates[0] - init_term_window + 1
            window_troughs = local_minimum[(local_minimum >= event_end) & (local_minimum <= end_of_window)]
            if len(window_troughs) > 0:
                closest_trough = np.abs(window_troughs - possible_end_day).argmin() #Finds the closest trough to the first peak
                end_day = window_troughs[closest_trough]

        else: #If termination conditions are not met for a bloom, find the next trough that is below the threshold value and make that the termination date.
                potential_trough = local_minimum[(local_minimum >= event_end) & (local_minimum <= end_of_window)]
                if len(potential_trough) > 0:
                    end_day = potential_trough[0]

    # STEP 6: Find peak DOY, date, and chlorophyll values for each bloom event.
        peak_date, peak_DOY, max_chl = max_peak(event,dataset,var_name_smooth=var_name_smooth)

    # STEP 7: Find the maximum rate of change for the bloom event
        max_roc = max_roc_for_bloom(start_DOY=start_day,end_DOY=end_day,dataset=dataset,roc_var_name=roc_var_name)

    # STEP 8: Merge and/or append events to the lists
        same_bloom = len(start_DOY) > 0 and start_day == start_DOY[-1] and end_day == end_DOY[-1]
        same_timing = (last_end_day>0) and (event[0]<=last_end_day)
        if same_bloom or same_timing: #This ensures that termination dates are not duplicated and every initiation date has a termination date
            merge_bloom_events[-1].extend(event)
            if end_day>end_DOY[-1]:
                end_DOY[-1] = end_day
                if max_chl > maximum_chl[-1]:
                    maximum_chl[-1] = max_chl
                    peak_dates[-1] = peak_date
                    peak_DOYs[-1] = peak_DOY
                #Recalculate the maximum ROC for the expanded timeline
                new_max_roc = max_roc_for_bloom(start_DOY=start_DOY[-1],end_DOY=end_DOY[-1],dataset=dataset)
                maximum_roc[-1] = new_max_roc

        else:
            #Documented as an entirely new event
            start_DOY.append(start_day)
            end_DOY.append(end_day)
            merge_bloom_events.append(list(event))
            peak_dates.append(peak_date)
            peak_DOYs.append(peak_DOY)
            maximum_chl.append(max_chl)
            maximum_roc.append(max_roc)
        
        last_start_day = start_day #Resets start and end dates for the loop
        last_end_day = end_day

    # STEP 9: Find the days where the chl first exceeds the threshold and where it last dips below the threshold
    chl_median_values = chl_median
    start_at_thld_list = []
    end_at_thld_list = []
    for i in range(len(start_DOY)):
        start = start_DOY[i]
        peak_start = merge_bloom_events[i][0]
        #Look forward for the day where it first crosses above the threshold
        start_found = False
        for j in range(start + 1, peak_start + 1, 1):
            if chl_median_values[j] >= clipped_thld:
                start_at_thld_list.append(j)
                start_found = True
                break
        if not start_found:
            start_at_thld_list.append(peak_start)

        end = end_DOY[i]
        peak_end = merge_bloom_events[i][-1]
        #Look backwards for the last drop below the threshold
        end_found = False
        if end > peak_end:
            for j in range (end, peak_end - 1, -1):
                if chl_median_values[j] >= clipped_thld:
                    end_at_thld_list.append(j+1)
                    end_found = True
                    break
        if not end_found:
            end_at_thld_list.append(end)
    return start_DOY,end_DOY,merge_bloom_events,peak_dates,peak_DOYs,maximum_chl, maximum_roc,start_at_thld_list,end_at_thld_list

To verify the characteristics produced, we plot them on a time series graph of chlorophyll to visually verify.

In [ ]:
time_series = [1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025]
title = ["Middle Atlantic Bight South","Middle Atlantic Bight North","Georges Bank","Gulf of Maine West","Gulf of Maine East"]
file = ["MABS","MABN","GB","GOMW","GOME"]
data_regions = [MABS,MABN,GB,GOMW,GOME]
shapefile_location = [MAB_south_loc,MAB_north_loc,GB_whole_loc,GOM_west_loc,GOM_east_loc]
clipped = [MABS_thld,MABN_thld,GB_thld,GOMW_thld,GOME_thld]
clipped_median = [MABS_median,MABN_median,GB_median,GOMW_median,GOME_median]
import time
for x in range(5):
    print(f"--- Starting Region {title[x]} ---")
    region_title = title[x]
    raw_data=data_regions[x]
    t0 = time.time()
    start_DOY,end_DOY,bloom_events,_,_,_,max_roc,_,_ = bloom_timing(clipped_thld=clipped[x],clipped_med=clipped_median[x],dataset=data_regions[x])
    t1 = time.time()
    smoothed = raw_data
    raw_time = pd.to_datetime(raw_data['time'].values)
    time_smoothed = smoothed['time']
    time_smoothed = pd.to_datetime(time_smoothed)
    chl_smoothed = smoothed['smoothed_sg_win_15_poly_3']
    roc = smoothed['ROC_SG']
    raw_chl = raw_data['CHL_median'].values

    #Converting ROC day of year into date format
    roc_max_time = np.asarray(max_roc).astype(int)
    roc_max_date = pd.to_datetime(time_smoothed[roc_max_time])
    roc_max = chl_smoothed[roc_max_time]

    #Bloom start and end date
    start_DOY_time = np.asarray(start_DOY).flatten().astype(int)
    start_DOY_date = pd.to_datetime(time_smoothed[start_DOY_time])
    start_DOY_val = chl_smoothed[start_DOY_time]
    end_DOY_time = np.asarray(end_DOY).flatten().astype(int)
    end_DOY_date = pd.to_datetime(time_smoothed[end_DOY_time])
    end_DOY_val = chl_smoothed[end_DOY_time]
    bloom_peak_time = np.concatenate(bloom_events if len(bloom_events)>0 else [np.asarray([])]).astype(int)
    bloom_peak_date = pd.to_datetime(time_smoothed[bloom_peak_time])
    bloom_peak_val = chl_smoothed[bloom_peak_time]
    
    for year in time_series:
        #print(f"  Plotting {year}...")
        x_left = year-1
        x_right = year+1
        start_bound = pd.to_datetime(f'{x_left}-01-01')
        end_bound = pd.to_datetime(f'{x_right}-01-01')

        mask_raw = (time_smoothed >= start_bound) & (time_smoothed < end_bound)
        mask_smooth = (time_smoothed >= start_bound) & (time_smoothed < end_bound)
        mask_roc = (roc_max_date >= start_bound) & (roc_max_date < end_bound)
        mask_start = (start_DOY_date >= start_bound) & (start_DOY_date < end_bound)
        mask_end = (end_DOY_date >= start_bound) & (end_DOY_date < end_bound)
        mask_peak = (bloom_peak_date >= start_bound) & (bloom_peak_date < end_bound)

        #Plotting
        fig=plt.figure(figsize=(18,10))
        plt.plot(raw_time[mask_raw].values,raw_chl[mask_raw],label="Raw Chl-a") #Plots the time series of raw data
        plt.plot(time_smoothed[mask_smooth],chl_smoothed[mask_smooth], label="Smoothed Chl-a")
        plt.axhline(clipped_median[x],c="purple",label="Climatological median")
        plt.axhline(clipped[x],c="red",label="10% Threshold")
        plt.scatter(roc_max_date[mask_roc],roc_max[mask_roc],c='green',s=100,zorder=5,marker='^',label="Max rates of change")
        plt.ylabel("Chlorophyll a Concentrations ($mg/m^3$)")
        plt.xlabel("Date")
        plt.xlim(start_bound, end_bound)
        plt.title("Chlorophyll a in the " + region_title + " in " + str(year))
        plt.scatter(start_DOY_date[mask_start],start_DOY_val[mask_start],c='magenta',s=50,zorder=5,marker='s',label="Bloom start")
        plt.scatter(end_DOY_date[mask_end],end_DOY_val[mask_end],c='darkblue',s=75,zorder=5,marker='*',label="Bloom end")
        plt.scatter(bloom_peak_date[mask_peak],bloom_peak_val[mask_peak],c='crimson',s=50,zorder=5,marker='D',label="Bloom peak")
        plt.legend(fontsize=8)

        axes_flat[1].plot(pd.to_datetime(time_smoothed),roc)
        roc_max_values = roc[roc_max_time]
        y=0
        axes_flat[1].scatter(roc_max_date[mask_roc],roc_max_values[mask_roc],c='green',s=100,zorder=5,marker='^',label="Max rates of change")
        axes_flat[1].axhline(y,c='gold')
        axes_flat[1].set_xlim(start_bound, end_bound)
        axes_flat[1].set_title("Rate of Change for " + region_title + " in " + str(year))
        axes_flat[1].legend()
        axes_flat[1].set_ylabel("Rate of Change per day")
        axes_flat[1].set_xlabel("Date")

        filename = f"{str(file[x])}_{str(year)}_5.png"
        plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\5day_med_graphs\{filename}')
        plt.close(fig)
    print(f"Successfully finished {file[x]} region graphs")

After testing both methods, we have decided that a combination of both most accurately captures our blooms on the Northeast US Shelf. Chlorophyll must surpass the 10% threshold to be considered an event and then we use the inflection points to identify initiation and termination dates.